In [16]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define Paths (Using raw strings to avoid errors)
train_dir = r'C:\Users\Arnav\Downloads\ASL\asl_alphabet_train\asl_alphabet_train'
test_dir = r"C:\Users\Arnav\Downloads\ASL\asl_alphabet_test"

# 2. Preprocessing for Training and Validation
# We use validation_split because the test set is reserved for final evaluation
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training' # 80% of data
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation' # 20% of data
)

# 3. Preprocessing for the Test Set
# Note: We do NOT use validation_split here; we want the whole test set.
# Modern approach
# 1. Re-define the test set using the modern API
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels=None,           # Crucial: This ignores the lack of subfolders
    image_size=(64, 64),
    batch_size=1,
    shuffle=False
)

# 2. Rescale the images (since your model was trained on rescaled 1./255 data)
test_ds = test_ds.map(lambda x: x / 255.0)

# 3. Generate Predictions
predictions = model.predict(test_ds)

Found 69600 images belonging to 29 classes.
Found 17400 images belonging to 29 classes.
Found 29 files.
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step  


In [17]:
from tensorflow.keras import layers, models

# The project requires detecting 29 classes 
NUM_CLASSES = 29

def build_asl_model():
    model = models.Sequential([
        # Convolutional layers to extract features from ASL signs
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Flattening and Dense layers for classification
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5), # Prevents overfitting
        layers.Dense(NUM_CLASSES, activation='softmax') # 29 output classes [cite: 10]
    ])
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_asl_model()
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)                    │ (None, 62, 62, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_7 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_8 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_8 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 29)                  │           3,741 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 686,941 (2.62 MB)

 Trainable params: 686,941 (2.62 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 1. Training the model
# The model learns from the train dataset folders [cite: 8, 9]
model.fit(
    train_generator,
    epochs=2,
    validation_data=validation_generator
)

# 1. Generate Predictions
# This works even if class_mode=None because it doesn't look for labels
predictions = model.predict(test_generator)

# 2. Get the Class Names (A-Z, SPACE, DELETE, NOTHING)
# We use the labels learned from the training set
class_labels = list(train_generator.class_indices.keys())

# 3. Show Results
import numpy as np
print(f"Found {len(predictions)} images to predict.")

for i, pred in enumerate(predictions):
    predicted_index = np.argmax(pred)
    actual_letter = class_labels[predicted_index]
    # Optionally: print only the first 10 to save space
    if i < 10:
        print(f"Image {i+1}: Predicted Sign -> {actual_letter}")

# 4. SAVE THE MODEL NOW
model.save('asl_detection_model.h5')
print("Model saved as 'asl_detection_model.h5'")

Epoch 1/2
 500/2175 ━━━━━━━━━━━━━━━━━━━━ 3:56 141ms/step - accuracy: 0.1427 - loss: 3.0116

In [5]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image

# 1. Path to your test images
test_path = r"C:\Users\Arnav\Downloads\ASL\asl_alphabet_test"

# 2. Get the list of class names from your successful training generator
# This ensures we have all 29 classes: A-Z, SPACE, DELETE, NOTHING [cite: 6]
class_labels = list(train_generator.class_indices.keys())

# 3. List all images in the test folder
test_images = [f for f in os.listdir(test_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

print(f"Checking {len(test_images)} test images against 29 classes...")

for img_name in test_images:
    img_path = os.path.join(test_path, img_name)
    
    # Preprocess the image to match the model's input
    img = image.load_img(img_path, target_size=(64, 64))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Make the prediction
    prediction = model.predict(img_array, verbose=0)
    predicted_index = np.argmax(prediction)
    predicted_label = class_labels[predicted_index]
    confidence = np.max(prediction)
    
    print(f"Image: {img_name:20} | Predicted: {predicted_label:10} | Confidence: {confidence*100:.2f}%")

Checking 29 test images against 29 classes...
Image: A_test.jpg           | Predicted: A          | Confidence: 100.00%
Image: B_test.jpg           | Predicted: B          | Confidence: 100.00%
Image: C_test.jpg           | Predicted: C          | Confidence: 100.00%
Image: del_test.jpg         | Predicted: del        | Confidence: 100.00%
Image: D_test.jpg           | Predicted: D          | Confidence: 100.00%
Image: E_test.jpg           | Predicted: E          | Confidence: 100.00%
Image: F_test.jpg           | Predicted: F          | Confidence: 100.00%
Image: G_test.jpg           | Predicted: G          | Confidence: 100.00%
Image: H_test.jpg           | Predicted: H          | Confidence: 100.00%
Image: I_test.jpg           | Predicted: I          | Confidence: 100.00%
Image: J_test.jpg           | Predicted: J          | Confidence: 100.00%
Image: K_test.jpg           | Predicted: K          | Confidence: 100.00%
Image: L_test.jpg           | Predicted: L          | Confidence: 

In [6]:
#import numpy as np
#from tensorflow.keras.preprocessing import image

#def predict_sign(img_path):
    # Map indices back to class names [cite: 6]
#    class_labels = list(train_generator.class_indices.keys())
    
#    img = image.load_img(img_path, target_size=IMAGE_SIZE)
#    img_array = image.img_to_array(img) / 255.0
#    img_array = np.expand_dims(img_array, axis=0)

#    prediction = model.predict(img_array)
#    result_index = np.argmax(prediction)
    
#    return class_labels[result_index]

# Example usage:
# print(f"Detected Sign: {predict_sign('test_image.jpg')}")

In [10]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Load your trained model
model = load_model('asl_detection_model.h5')

# 2. Define the 29 classes in the correct alphabetical order
# (A-Z, then del, nothing, space based on folder sorting)
labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
          'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
          'del', 'nothing', 'space']

# 3. Initialize Webcam
cap = cv2.VideoCapture(0)

print("Press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Mirror the frame for easier interaction
    frame = cv2.flip(frame, 1)
    
    # Define the Region of Interest (ROI) box coordinates
    # We will detect the hand inside this square
    x1, y1, x2, y2 = 100, 100, 400, 400
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
    
    # Extract and preprocess the ROI
    roi = frame[y1:y2, x1:x2]
    img = cv2.resize(roi, (64, 64)) # Match model input size
    img = img / 255.0               # Normalize
    img = np.expand_dims(img, axis=0)
    
    # 4. Make Prediction
    prediction = model.predict(img, verbose=0)
    index = np.argmax(prediction)
    confidence = np.max(prediction)
    
    predicted_label = labels[index]
    
    # 5. Display the result on the screen
    display_text = f"Sign: {predicted_label} ({confidence*100:.1f}%)"
    cv2.putText(frame, display_text, (100, 80), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow("ASL Real-Time Detection", frame)
    
    # Exit loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'q' to quit.
